# FuelGuard IoT Scenarios Notebook

This notebook simulates all required operating scenarios for a hardware-style Fuel Theft Detection System using ignition state, cap event, vibration event, periodic wake-up, LSTM anomaly output, and GSM alert logic.

## Scenario Coverage

1. Vehicle ON (Driving / Engine Running)
2. Vehicle OFF (Normal Idle)
3. Refueling at Petrol Bunk
4. Fuel Theft (Cap Open / Siphoning)
5. Tank Puncture / No Cap Event
6. No Activity (Deep Sleep)

Power management states are also tracked:
- Deep Sleep: Very Low
- Event Wake: Medium
- GSM Transmission: High

In [1]:
from dataclasses import dataclass, field
from datetime import datetime
from typing import Dict, List, Optional
import random

In [2]:
@dataclass
class Telemetry:
    ignition_on: bool
    fuel_rate_lph: float
    fuel_level_percent: float
    fuel_level_later_percent: Optional[float] = None
    cap_open: bool = False
    vibration: bool = False
    timer_wake: bool = False
    lstm_anomaly_score: float = 0.0


@dataclass
class Decision:
    scenario: str
    mode: str
    power_state: str
    event_summary: str
    result: str
    alert: bool
    alert_channel: str
    details: Dict[str, float] = field(default_factory=dict)
    ts: str = field(default_factory=lambda: datetime.now().strftime('%Y-%m-%d %H:%M:%S'))

In [3]:
class FuelGuardIoTSimulator:
    """
    Hardware-style IoT simulation logic for the six scenarios.
    - Vehicle ON uses built-in sensor + LSTM anomaly detection.
    - Vehicle OFF uses ESP32 active monitoring with deep sleep cycles.
    - Refueling/theft decisions depend on cap/vibration events and fuel delta.
    """

    def __init__(self, theft_drop_threshold=2.0, lstm_alert_threshold=0.65):
        self.theft_drop_threshold = theft_drop_threshold
        self.lstm_alert_threshold = lstm_alert_threshold
        self.log: List[Decision] = []

    def _push(self, decision: Decision) -> Decision:
        self.log.append(decision)
        return decision

    def scenario_1_vehicle_on(self, t: Telemetry) -> Decision:
        # Vehicle ON: ESP32 standby, built-in fuel sensor + LSTM active.
        mode = "ESP32 standby | Vehicle ECU path active"
        power_state = "Deep Sleep (Very Low)"

        anomaly = t.lstm_anomaly_score >= self.lstm_alert_threshold
        if anomaly:
            result = "Abnormal consumption / leak / sensor anomaly"
            alert = True
            alert_channel = "GSM SMS + Server"
            event_summary = "LSTM anomaly in driving state"
            power_state = "GSM Transmission (High)"
        else:
            result = "Normal driving consumption"
            alert = False
            alert_channel = "None"
            event_summary = "Continuous monitoring during ignition ON"

        return self._push(Decision(
            scenario="SCENARIO 1: Vehicle ON",
            mode=mode,
            power_state=power_state,
            event_summary=event_summary,
            result=result,
            alert=alert,
            alert_channel=alert_channel,
            details={
                "fuel_rate_lph": t.fuel_rate_lph,
                "fuel_level_percent": t.fuel_level_percent,
                "lstm_anomaly_score": t.lstm_anomaly_score
            }
        ))

    def scenario_2_vehicle_off_idle(self, t: Telemetry) -> Decision:
        # Vehicle OFF: ESP32 active monitoring on 18650 battery with deep sleep cycle.
        mode = "ESP32 active monitoring on battery"
        power_state = "Deep Sleep (Very Low) -> Event Wake (Medium) -> Deep Sleep"
        event_summary = "Sleep -> Wake -> Measure -> Decide -> Sleep"

        result = "No suspicious drop in idle monitoring"
        alert = False
        alert_channel = "None"

        return self._push(Decision(
            scenario="SCENARIO 2: Vehicle OFF (Normal Idle)",
            mode=mode,
            power_state=power_state,
            event_summary=event_summary,
            result=result,
            alert=alert,
            alert_channel=alert_channel,
            details={
                "fuel_level_percent": t.fuel_level_percent
            }
        ))

    def scenario_3_refueling(self, t: Telemetry) -> Decision:
        mode = "ESP32 event mode (cap-open wake)"
        power_state = "Event Wake (Medium)"

        if t.fuel_level_later_percent is None:
            raise ValueError("fuel_level_later_percent is required for refueling scenario")

        t1 = t.fuel_level_percent
        t2 = t.fuel_level_later_percent

        if t.cap_open and t2 > t1:
            result = "Normal refueling confirmed"
            alert = False
            alert_channel = "None"
            event_summary = "Cap-open event and fuel increased"
        else:
            result = "Refueling pattern not confirmed"
            alert = False
            alert_channel = "None"
            event_summary = "Event captured, but no positive refuel delta"

        return self._push(Decision(
            scenario="SCENARIO 3: Refueling at Petrol Bunk",
            mode=mode,
            power_state=power_state,
            event_summary=event_summary,
            result=result,
            alert=alert,
            alert_channel=alert_channel,
            details={
                "T1": t1,
                "T2": t2,
                "delta": t2 - t1
            }
        ))

    def scenario_4_fuel_theft_cap_or_siphon(self, t: Telemetry) -> Decision:
        mode = "ESP32 event mode (cap/vibration wake)"
        if t.fuel_level_later_percent is None:
            raise ValueError("fuel_level_later_percent is required for theft scenario")

        t1 = t.fuel_level_percent
        t2 = t.fuel_level_later_percent
        drop = t1 - t2

        theft = (t.cap_open or t.vibration) and (drop > self.theft_drop_threshold)

        if theft:
            result = "Fuel theft detected"
            alert = True
            alert_channel = "GSM SMS + Server"
            power_state = "GSM Transmission (High)"
            event_summary = "Cap/vibration event with drop beyond threshold"
        else:
            result = "No confirmed theft in cap/vibration event"
            alert = False
            alert_channel = "None"
            power_state = "Event Wake (Medium)"
            event_summary = "Event observed but drop below threshold"

        return self._push(Decision(
            scenario="SCENARIO 4: Fuel Theft (Cap Open / Siphoning)",
            mode=mode,
            power_state=power_state,
            event_summary=event_summary,
            result=result,
            alert=alert,
            alert_channel=alert_channel,
            details={
                "T1": t1,
                "T2": t2,
                "drop": drop,
                "drop_threshold": self.theft_drop_threshold
            }
        ))

    def scenario_5_tank_puncture_no_cap(self, t: Telemetry) -> Decision:
        mode = "ESP32 periodic wake / vibration wake"
        if t.fuel_level_later_percent is None:
            raise ValueError("fuel_level_later_percent is required for puncture scenario")

        t1 = t.fuel_level_percent
        t2 = t.fuel_level_later_percent
        drop = t1 - t2

        theft_or_tamper = (not t.cap_open) and (t.vibration or t.timer_wake) and (drop > self.theft_drop_threshold)

        if theft_or_tamper:
            result = "Theft/tampering detected (no cap event)"
            alert = True
            alert_channel = "GSM SMS + Server"
            power_state = "GSM Transmission (High)"
            event_summary = "Unexpected drop without cap event"
        else:
            result = "No puncture/tamper confirmation"
            alert = False
            alert_channel = "None"
            power_state = "Event Wake (Medium)"
            event_summary = "Periodic/vibration check completed"

        return self._push(Decision(
            scenario="SCENARIO 5: Tank Puncture / No Cap Event",
            mode=mode,
            power_state=power_state,
            event_summary=event_summary,
            result=result,
            alert=alert,
            alert_channel=alert_channel,
            details={
                "T1": t1,
                "T2": t2,
                "drop": drop,
                "drop_threshold": self.theft_drop_threshold
            }
        ))

    def scenario_6_no_activity(self) -> Decision:
        mode = "ESP32 deep sleep"
        power_state = "Deep Sleep (Very Low)"
        event_summary = "Wake only on reed switch / vibration / timer interrupt"
        result = "No activity; battery preserved"

        return self._push(Decision(
            scenario="SCENARIO 6: No Activity",
            mode=mode,
            power_state=power_state,
            event_summary=event_summary,
            result=result,
            alert=False,
            alert_channel="None"
        ))

    def run_all_demo(self) -> List[Decision]:
        # Scenario 1: Vehicle ON with a random LSTM score in realistic range.
        s1_score = round(random.uniform(0.35, 0.92), 3)
        self.scenario_1_vehicle_on(Telemetry(
            ignition_on=True,
            fuel_rate_lph=6.8,
            fuel_level_percent=62.4,
            lstm_anomaly_score=s1_score
        ))

        # Scenario 2: Vehicle OFF normal idle.
        self.scenario_2_vehicle_off_idle(Telemetry(
            ignition_on=False,
            fuel_rate_lph=0.0,
            fuel_level_percent=61.9
        ))

        # Scenario 3: Refueling event.
        self.scenario_3_refueling(Telemetry(
            ignition_on=False,
            fuel_rate_lph=0.0,
            fuel_level_percent=28.2,
            fuel_level_later_percent=41.7,
            cap_open=True
        ))

        # Scenario 4: Cap-open siphoning theft.
        self.scenario_4_fuel_theft_cap_or_siphon(Telemetry(
            ignition_on=False,
            fuel_rate_lph=0.0,
            fuel_level_percent=54.6,
            fuel_level_later_percent=50.9,
            cap_open=True,
            vibration=True
        ))

        # Scenario 5: Puncture/no-cap event.
        self.scenario_5_tank_puncture_no_cap(Telemetry(
            ignition_on=False,
            fuel_rate_lph=0.0,
            fuel_level_percent=49.1,
            fuel_level_later_percent=45.8,
            cap_open=False,
            vibration=True,
            timer_wake=True
        ))

        # Scenario 6: No activity state.
        self.scenario_6_no_activity()

        return self.log

In [4]:
sim = FuelGuardIoTSimulator(theft_drop_threshold=2.0, lstm_alert_threshold=0.65)
results = sim.run_all_demo()

for i, d in enumerate(results, start=1):
    print(f"\n[{i}] {d.scenario}")
    print(f"  Mode        : {d.mode}")
    print(f"  Power State : {d.power_state}")
    print(f"  Event       : {d.event_summary}")
    print(f"  Result      : {d.result}")
    print(f"  Alert       : {'YES' if d.alert else 'NO'}")
    print(f"  Channel     : {d.alert_channel}")
    if d.details:
        print(f"  Details     : {d.details}")


[1] SCENARIO 1: Vehicle ON
  Mode        : ESP32 standby | Vehicle ECU path active
  Power State : GSM Transmission (High)
  Event       : LSTM anomaly in driving state
  Result      : Abnormal consumption / leak / sensor anomaly
  Alert       : YES
  Channel     : GSM SMS + Server
  Details     : {'fuel_rate_lph': 6.8, 'fuel_level_percent': 62.4, 'lstm_anomaly_score': 0.698}

[2] SCENARIO 2: Vehicle OFF (Normal Idle)
  Mode        : ESP32 active monitoring on battery
  Power State : Deep Sleep (Very Low) -> Event Wake (Medium) -> Deep Sleep
  Event       : Sleep -> Wake -> Measure -> Decide -> Sleep
  Result      : No suspicious drop in idle monitoring
  Alert       : NO
  Channel     : None
  Details     : {'fuel_level_percent': 61.9}

[3] SCENARIO 3: Refueling at Petrol Bunk
  Mode        : ESP32 event mode (cap-open wake)
  Power State : Event Wake (Medium)
  Event       : Cap-open event and fuel increased
  Result      : Normal refueling confirmed
  Alert       : NO
  Channel    

In [5]:
import pandas as pd

summary_df = pd.DataFrame([
    {
        "Scenario": d.scenario,
        "Mode": d.mode,
        "Power Usage State": d.power_state,
        "Decision": d.result,
        "Alert": "YES" if d.alert else "NO",
        "Alert Channel": d.alert_channel
    }
    for d in results
])

summary_df

,Scenario,Mode,Power Usage State,Decision,Alert,Alert Channel
0,SCENARIO 1: Vehicle ON,ESP32 standby | Vehicle ECU path active,GSM Transmission (High),Abnormal consumption / leak / sensor anomaly,YES,GSM SMS + Server
1,SCENARIO 2: Vehicle OFF (Normal Idle),ESP32 active monitoring on battery,Deep Sleep (Very Low) -> Event Wake (Medium) -...,No suspicious drop in idle monitoring,NO,None
2,SCENARIO 3: Refueling at Petrol Bunk,ESP32 event mode (cap-open wake),Event Wake (Medium),Normal refueling confirmed,NO,None
3,SCENARIO 4: Fuel Theft (Cap Open / Siphoning),ESP32 event mode (cap/vibration wake),GSM Transmission (High),Fuel theft detected,YES,GSM SMS + Server
4,SCENARIO 5: Tank Puncture / No Cap Event,ESP32 periodic wake / vibration wake,GSM Transmission (High),Theft/tampering detected (no cap event),YES,GSM SMS + Server
5,SCENARIO 6: No Activity,ESP32 deep sleep,Deep Sleep (Very Low),No activity; battery preserved,NO,None


## Power Management Logic

| Mode | Power Usage |
|---|---|
| Deep Sleep | Very Low |
| Event Wake | Medium |
| GSM Transmission | High |

In this simulation:
- No activity remains in Deep Sleep.
- Sensor/cap/vibration/timer events trigger Event Wake.
- Confirmed anomaly/theft triggers GSM Transmission.